<a href="https://colab.research.google.com/github/BohdanKozak/UA-Toponyms-Dataset-Creation/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import urllib.request
import zipfile
import shutil
import matplotlib.pyplot as plt

# 1. Створюємо правильну структуру папок для здачі
base_dir = "lab1_submission"
dirs = [f"{base_dir}/data", f"{base_dir}/docs", f"{base_dir}/notebooks"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Папки створено. Завантажуємо відкриті дані GeoNames (UA)...")

# 2. Завантаження даних GeoNames
url = "https://download.geonames.org/export/dump/UA.zip"
urllib.request.urlretrieve(url, "UA.zip")
with zipfile.ZipFile("UA.zip", 'r') as zip_ref:
    zip_ref.extractall(".")

# GeoNames формат (tab-separated)
columns = [
    'geonameid', 'name', 'asciiname', 'alternatenames', 'latitude', 'longitude',
    'feature_class', 'feature_code', 'country_code', 'cc2', 'admin1_code',
    'admin2_code', 'admin3_code', 'admin4_code', 'population', 'elevation',
    'dem', 'timezone', 'modification_date'
]

# Завантажуємо дані. Беремо лише населені пункти (feature_class == 'P') для чистоти датасету
df_raw = pd.read_csv('UA.txt', sep='\t', header=None, names=columns, dtype=str)
df_raw = df_raw[df_raw['feature_class'] == 'P'].copy()

# Зберігаємо сирі дані у вимогах лаби (збережемо лише потрібні колонки щоб файл не був гігантським)
df_raw_subset = df_raw[['geonameid', 'name', 'alternatenames']].dropna(subset=['alternatenames'])
df_raw_subset.to_csv(f"{base_dir}/data/raw.csv", index=False)
print(f"Завантажено {len(df_raw_subset)} базових локацій з альтернативними назвами.")

Папки створено. Завантажуємо відкриті дані GeoNames (UA)...
Завантажено 29988 базових локацій з альтернативними назвами.


In [ ]:
# Розпаковуємо альтернативні назви (comma-separated) у список, а потім у рядки
records = []
for _, row in df_raw_subset.iterrows():
    canonical = str(row['name'])
    geo_id = row['geonameid']
    # Розбиваємо альтернативні назви
    alts = str(row['alternatenames']).split(',')
    for alt in alts:
        alt = alt.strip()
        if alt and alt != canonical: # беремо лише ті, що відрізняються від канонічного
            records.append({
                'text_id': f"{geo_id}_{len(records)}",
                'surface_form': alt,
                'canonical_name': canonical
            })

df_dataset = pd.DataFrame(records)

print("Трансформацію завершено. 10 випадкових прикладів:")
display(df_dataset.sample(10))

Трансформацію завершено. 10 випадкових прикладів:


,text_id,surface_form,canonical_name
82607,709233_82607,Зубакіне,Zubakino
83070,709359_83070,Mali Didushichi,Mali Didushychi
43533,698591_43533,Olijniki,Motuzivka
73444,706685_73444,Shirokoe,Shafranne
93582,712160_93582,Axkʻirman,Bilhorod-Dnistrovskyi
72365,706381_72365,Khmel'naja,Khmilna
93968,712261_93968,Beremovtse,Berymivtsi
36,490588_36,索皮奇,Sopych
69126,705442_69126,Kulewcza,Kulevcha
95238,712587_95238,Karasu-Bazar,Bilohirsk


In [ ]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    # Заміна URL/email/phone (хоча в топонімах це рідкість, вимога є вимога)
    text = re.sub(r'http[s]?://\S+|www\.\S+', '<URL>', text)
    text = re.sub(r'\S+@\S+', '<EMAIL>', text)
    text = re.sub(r'\+?\d{10,12}', '<PHONE>', text)

    # Уніфікація апострофів (всі види '`’ замінюємо на правильний український ’)
    text = re.sub(r"['`‘´]", "’", text)

    # Прибрати зайві пробіли/переноси
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_dataset['surface_form_clean'] = df_dataset['surface_form'].apply(normalize_text)

# Зберігаємо оброблені дані
df_processed = df_dataset[['text_id', 'surface_form_clean', 'canonical_name']].rename(columns={'surface_form_clean': 'surface_form'})
df_processed.to_csv(f"{base_dir}/data/processed.csv", index=False)

# Зберігаємо labels.csv (в даному випадку surface_form -> label(canonical_name))
df_processed.to_csv(f"{base_dir}/data/labels.csv", index=False)
print("Дані нормалізовано та збережено.")

Дані нормалізовано та збережено.


In [ ]:
print("--- АУДИТ ДАНИХ ---")

# 1. Кількість текстів
total_texts = len(df_processed)
print(f"1. Загальна кількість пар (surface form -> canonical): {total_texts}")

# 2. Довжина (символи + слова)
df_processed['char_len'] = df_processed['surface_form'].apply(len)
df_processed['word_count'] = df_processed['surface_form'].apply(lambda x: len(x.split()))

print(f"2. Довжина surface_form:")
print(f"   - Середня кількість символів: {df_processed['char_len'].mean():.2f}")
print(f"   - Медіанна кількість символів: {df_processed['char_len'].median()}")
print(f"   - Середня кількість слів: {df_processed['word_count'].mean():.2f}")
print(f"   - Медіанна кількість слів: {df_processed['word_count'].median()}")

# 3. Розподіл класів (Топ-5 населених пунктів з найбільшою кількістю варіацій)
print("\n3. Топ-5 локацій за кількістю варіантів написання (ваші 'класи'):")
print(df_processed['canonical_name'].value_counts().head(5))

# 4. Перевірки якості (Light)
# Дублі точні
exact_duplicates = df_processed.duplicated(subset=['surface_form', 'canonical_name']).sum()
dup_percent = (exact_duplicates / total_texts) * 100
print(f"\n4. Перевірки якості:")
print(f"   - Точні дублікати: {exact_duplicates} ({dup_percent:.2f}%)")

# Дуже короткі/порожні (менше 3 символів, бо для топонімів < 5 слів це норма, а от 1-2 букви - це часто шум)
short_texts = df_processed[df_processed['char_len'] < 3]
print(f"   - Дуже короткі топоніми (< 3 символів): {len(short_texts)} ({(len(short_texts)/total_texts)*100:.2f}%)")

# "Сміттєві" рядки (тільки цифри)
digits_only = df_processed[df_processed['surface_form'].str.match(r'^\d+$')].shape[0]
print(f"   - Сміттєві рядки (лише цифри): {digits_only}")

print("\n--- ВИСНОВОК ---")

--- АУДИТ ДАНИХ ---
1. Загальна кількість пар (surface form -> canonical): 132712
2. Довжина surface_form:
   - Середня кількість символів: 9.92
   - Медіанна кількість символів: 9.0
   - Середня кількість слів: 1.16
   - Медіанна кількість слів: 1.0

3. Топ-5 локацій за кількістю варіантів написання (ваші 'класи'):
canonical_name
Mykolaivka     416
Mykhailivka    379
Vyshneve       354
Vasylivka      330
Kalynivka      323
Name: count, dtype: int64

4. Перевірки якості:
   - Точні дублікати: 35105 (26.45%)
   - Дуже короткі топоніми (< 3 символів): 58 (0.04%)
   - Сміттєві рядки (лише цифри): 0

--- ВИСНОВОК ---

Цей датасет побудовано на основі відкритих даних GeoNames для населених пунктів України. 
Загалом зібрано достатньо велику кількість альтернативних назв, що покривають транслітерації, історичні назви (напр. Lemberg) та різні мови. 
Основний ризик датасету — значна кількість іноземних (не українських і не англійських) написань, які можуть стати шумом для специфічної UA-задачі.

In [ ]:
# Очистимо тимчасові колонки
df_processed.drop(columns=['char_len', 'word_count'], inplace=True)

# Архівація
shutil.make_archive("lab1_submission", 'zip', "lab1_submission")

Готово! Архів lab1_submission.zip створено. Ви можете завантажити його з лівої панелі 'Files'.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')